# 03 · 手刻去噪 U-Net：訓練模型生成數字

整條軌道的**核心課**。前向加噪不用學;要學的是**反向去噪**。做法出奇地簡單:

> 訓練一個神經網路,**看著加噪圖 `x_t` 和時間步 `t`,把當初加進去的噪聲 `ε` 預測出來**。損失就是「預測的噪聲」和「真正的噪聲」的 MSE。

學會預測噪聲,等於學會了「這張圖該往哪個方向去噪」。我們用一個迷你 **U-Net** 來當這個去噪器。

## 1. 設定、模型、排程、資料

In [ ]:
%pip install -q -U torchvision

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalPosEmb(nn.Module):
    """把整數時間步 t 編碼成向量(同 Transformer 的位置編碼)。"""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock(nn.Module):
    """conv 殘差塊,並把時間嵌入加進特徵圖。"""

    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.temb = nn.Linear(tdim, cout)
        self.norm2 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.temb(temb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class TinyUNet(nn.Module):
    """最小可用的 U-Net:下採樣兩次、上採樣兩次,帶 skip 連接。"""

    def __init__(self, base=32, tdim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(tdim), nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim)
        )
        self.in_conv = nn.Conv2d(1, base, 3, padding=1)
        self.d1 = ResBlock(base, base, tdim)
        self.down1 = nn.Conv2d(base, base, 4, 2, 1)
        self.d2 = ResBlock(base, base * 2, tdim)
        self.down2 = nn.Conv2d(base * 2, base * 2, 4, 2, 1)
        self.mid = ResBlock(base * 2, base * 2, tdim)
        self.up2 = nn.ConvTranspose2d(base * 2, base * 2, 4, 2, 1)
        self.u2 = ResBlock(base * 2 + base * 2, base, tdim)
        self.up1 = nn.ConvTranspose2d(base, base, 4, 2, 1)
        self.u1 = ResBlock(base + base, base, tdim)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, 1, 3, padding=1))

    def forward(self, x, t):
        temb = self.time_mlp(t)
        x = self.in_conv(x)
        s1 = self.d1(x, temb)
        x = self.down1(s1)
        s2 = self.d2(x, temb)
        x = self.down2(s2)
        x = self.mid(x, temb)
        x = self.up2(x)
        x = self.u2(torch.cat([x, s2], dim=1), temb)
        x = self.up1(x)
        x = self.u1(torch.cat([x, s1], dim=1), temb)
        return self.out(x)

In [ ]:
import torch

T = 200                                                  # 擴散步數
betas = torch.linspace(1e-4, 0.02, T, device=device)     # 線性 beta 排程
alphas = 1.0 - betas
alphabars = torch.cumprod(alphas, dim=0)                  # 連乘:一步到位的關鍵


def q_sample(x0, t, noise):
    """前向加噪(closed form):x_t = sqrt(ab)·x0 + sqrt(1-ab)·noise。"""
    ab = alphabars[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise


@torch.no_grad()
def ddpm_sample(model, n=16):
    """反向去噪:從純噪聲一步步還原,共 T 步(慢但標準)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    for i in reversed(range(T)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        a, ab, beta = alphas[i], alphabars[i], betas[i]
        mean = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt()
        x = mean + beta.sqrt() * torch.randn_like(x) if i > 0 else mean
    return x


@torch.no_grad()
def ddim_sample(model, n=16, steps=20):
    """DDIM:確定性取樣,跳著走、只要 steps 步(快很多)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    seq = torch.linspace(T - 1, 0, steps, device=device).long().tolist()
    for j, i in enumerate(seq):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        ab = alphabars[i]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()         # 預測乾淨影像
        if j < len(seq) - 1:
            ab_next = alphabars[seq[j + 1]]
            x = ab_next.sqrt() * x0 + (1 - ab_next).sqrt() * eps
        else:
            x = x0
    return x

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

tf = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),     # → [-1, 1]
])
mnist = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
loader = DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2)
print(f"MNIST {len(mnist)} 張,縮到 32×32、正規化到 [-1,1]")

## 2. U-Net 去噪器

**U-Net** 是影像生成的標準骨架:下採樣抓全局、上採樣回細節,中間用 skip 連接保留細節。它額外吃一個**時間步 t**(用正弦編碼),才知道現在在去噪的哪個階段。

In [ ]:
model = TinyUNet().to(device)
print(sum(p.numel() for p in model.parameters()), "個參數")

## 3. 訓練:讓它學會預測噪聲

每一步:隨機抽 `t` → 加噪得 `x_t` → 模型預測噪聲 → 跟真噪聲算 MSE。MNIST 上幾個 epoch(T4 幾分鐘)就能生出像樣的數字。**功能不求強,重在跑通機制。**

In [ ]:
import torch
import torch.nn.functional as F


def train_diffusion(model, loader, epochs=5, lr=2e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        model.train()
        total, nb = 0.0, 0
        for x, _ in loader:
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            pred = model(q_sample(x, t, noise), t)        # 預測加進去的噪聲
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); nb += 1
        print(f"epoch {ep}/{epochs}  loss {total / nb:.4f}")
    return model

In [ ]:
model = train_diffusion(model, loader, epochs=5)

## 4. 生成!從純噪聲反向去噪

餵一批純雪花,跑完整個反向過程,看數字「長」出來。

In [ ]:
import matplotlib.pyplot as plt


def show_gen(imgs, cols=8, title=None):
    imgs = ((imgs.clamp(-1, 1) + 1) / 2).detach().cpu()    # [-1,1] → [0,1]
    n = len(imgs); rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 1.1, rows * 1.15))
    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(imgs[i, 0], cmap="gray"); ax.axis("off")
    if title:
        plt.suptitle(title)
    plt.tight_layout(); plt.show()

In [ ]:
samples = ddpm_sample(model, n=16)
show_gen(samples, cols=8, title="Generated from pure noise")

## 小結

- **去噪 = 訓練網路預測「加進去的噪聲」**,損失是預測噪聲與真噪聲的 MSE。
- **U-Net + 時間步嵌入**是擴散模型的標準去噪器。
- 訓練好後,**餵純噪聲、反覆去噪,就生出全新數字**——這就是擴散生成的本質。
- 你的數字也許歪歪扭扭,沒關係——**重點是你親手跑通了 Stable Diffusion 的核心機制**。

下一課:加速生成——比較 **DDPM(慢)與 DDIM(快)** 兩種取樣,並看去噪的逐步軌跡。